In [2]:
import pandas as pd

df = pd.read_csv("hf://datasets/wesley7137/question_complexity_classification/cleaned_output.csv")

# Example thresholds
def rating_to_class(rating):
    if rating <= 0.33:
        return 0  # Easy
    elif rating <= 0.66:
        return 1  # Medium
    else:
        return 2  # Hard

df['label'] = df['rating'].apply(rating_to_class)

c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['question'],  # replace 'text' with your column name
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
print(train_encodings.keys())

In [3]:
# pip install transformers datasets scikit-learn torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, pipeline

# ----------------------------
# Step 1: Prepare sample dataset with difficulty levels
# ----------------------------
data = {
    "query": [
        "What is a sorting algorithm?",                       # Easy
        "Explain recursion with examples",                     # Medium
        "How many permutations exist for 5 objects?",         # Medium
        "Prove the formula for the number of combinations",   # Hard
        "What is Pascal's Triangle?",                          # Easy
        "Derive the number of ways to assign n hats to n people", # Hard
        "Explain the difference between stack and queue",      # Easy
        "Write a Python function to generate all permutations of a list", # Hard
        "Compute factorial of 10 using recursion",             # Medium
        "Solve for X(t) probability in continuous-time markets" # Hard
    ],
    "label": [
        "Easy",
        "Medium",
        "Medium",
        "Hard",
        "Easy",
        "Hard",
        "Easy",
        "Hard",
        "Medium",
        "Hard"
    ]
}

# Train/test split
train_texts, test_texts, train_labels, test_labels = train_test_split(
    data["query"], data["label"], test_size=0.2, random_state=42
)

train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
test_dataset = Dataset.from_dict({"text": test_texts, "label": test_labels})

# Map string labels to integer ids
label_list = list(set(train_labels + test_labels))
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

train_dataset = train_dataset.map(lambda x: {"label": label2id[x["label"]]})
test_dataset = test_dataset.map(lambda x: {"label": label2id[x["label"]]})

# ----------------------------
# Step 2: Tokenization
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

# ----------------------------
# Step 3: Model setup
# ----------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list)
)

# ----------------------------
# Step 4: Training
# ----------------------------
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir='./logs',
    logging_steps=5
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

# ----------------------------
# Step 5: Test classifier
# ----------------------------
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)

# Example queries
test_queries = [
    "What is a sorting algorithm?",
    "Write a Python function to generate all permutations of a list",
    "Compute factorial of 10 using recursion"
]

for q in test_queries:
    print(f"Query: {q}")
    print(classifier(q))
    print()


Map: 100%|██████████| 2/2 [00:00<00:00, 460.31 examples/s]
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,1.242475
2,1.085600,1.334434
3,1.001000,1.430693
4,0.859100,1.532476
5,0.791300,1.587307


c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Migs\Desktop\ragbot\.venv\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Device set to use cpu
c:\Users\Migs\Desktop\ragbot\.venv\Lib

Query: What is a sorting algorithm?
[[{'label': 'LABEL_0', 'score': 0.359095960855484}, {'label': 'LABEL_1', 'score': 0.23586970567703247}, {'label': 'LABEL_2', 'score': 0.4050343334674835}]]

Query: Write a Python function to generate all permutations of a list
[[{'label': 'LABEL_0', 'score': 0.6222639679908752}, {'label': 'LABEL_1', 'score': 0.1929110735654831}, {'label': 'LABEL_2', 'score': 0.18482498824596405}]]

Query: Compute factorial of 10 using recursion
[[{'label': 'LABEL_0', 'score': 0.5969162583351135}, {'label': 'LABEL_1', 'score': 0.20428723096847534}, {'label': 'LABEL_2', 'score': 0.19879651069641113}]]



In [4]:
model_save_path = "./fine_tuned_difficulty_classifier"
trainer.save_model(model_save_path)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_save_path = "./fine_tuned_difficulty_classifier"

# Load the saved tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(
    "distilbert-base-uncased",
    local_files_only=False, # Ensure this is False (default) or explicitly set it
    # Optional: Set force_download=True to be aggressive
    force_download=True 
)

# Load the saved model (it automatically detects the configuration, including labels)
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_save_path)

# Create a new pipeline with the loaded model and tokenizer
new_classifier = pipeline(
    "text-classification",
    model=loaded_model,
    tokenizer=loaded_tokenizer,
    return_all_scores=True
)

# Test the loaded classifier
test_query = "Prove the formula for the number of combinations"
print(f"\nTest on loaded model: {test_query}")
print(new_classifier(test_query))

TypeError: _path_isfile: path should be string, bytes, os.PathLike or integer, not NoneType